In [1]:
import cv2
import mediapipe as mp
import numpy as np


In [2]:
from ultralytics import YOLO
yolo_model = YOLO('yolov8n.pt')

In [3]:
# results=yolo_model.train(data=r"C:\project\Boxing-Gloves-Detection--2\data.yaml",epochs=15)

yolo_model = YOLO(r"C:\project\runs\detect\train-10\weights\last.pt")
# results = yolo_model.train(resume=True)



In [4]:
#mediapipe
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=False,
    model_complexity=1,
    enable_segmentation=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5)
mp_drawing = mp.solutions.drawing_utils
# mp_drawing_styles = mp.solutions.drawing_styles

In [9]:
import time

prev_time = time.time()
# prev_theta_l = None
# prev_theta_r = None
# PUNCH_THRESHOLD = 300
# HOLD_FRAMES = 150
# peak_speed_l = 0.0 
# counter_l = 5

# peak_speed_r = 0.0
# counter_r = 5
PUNCH_START = 200
PUNCH_END = 100
DISPLAY_SECS = 1.5  

is_punching_l = False
curr_peak_l = 0.0
last_peak_l = 0.0
timer_l = 0.0

is_punching_r = False
curr_peak_r = 0.0
last_peak_r = 0.0
timer_r = 0.0

In [6]:
def calculate_angle(cords,eps=1e-6):
    l_shoulder=cords['l_shoulder']
    r_shoulder=cords['r_shoulder']
    l_elbow=cords['l_elbow']
    r_elbow=cords['r_elbow']
    l_wrist=cords['l_wrist']
    r_wrist=cords['r_wrist']
    l_hip=cords['l_hip']
    r_hip=cords['r_hip']

    a1_sh=l_hip-l_shoulder
    b1_sh=l_elbow-l_shoulder
    a1_el=l_shoulder-l_elbow
    b1_el=l_wrist-l_elbow

    a2_sh=r_hip-r_shoulder
    b2_sh=r_elbow-r_shoulder
    a2_el=r_shoulder-r_elbow
    b2_el=r_wrist-r_elbow

    cos1_sh=np.dot(a1_sh,b1_sh)/(np.linalg.norm(a1_sh)*np.linalg.norm(b1_sh)+eps)
    cos1_el=np.dot(a1_el,b1_el)/(np.linalg.norm(a1_el)*np.linalg.norm(b1_el)+eps)
    cos2_sh=np.dot(a2_sh,b2_sh)/(np.linalg.norm(a2_sh)*np.linalg.norm(b2_sh)+eps)
    cos2_el=np.dot(a2_el,b2_el)/(np.linalg.norm(a2_el)*np.linalg.norm(b2_el)+eps)
    
    angle1_l=np.arccos(np.clip(cos1_sh,-1.0,1.0))
    angle2_l=np.arccos(np.clip(cos1_el,-1.0,1.0))
    angle1_r=np.arccos(np.clip(cos2_sh,-1.0,1.0))
    angle2_r=np.arccos(np.clip(cos2_el,-1.0,1.0))
    return (angle1_l, angle2_l), (angle1_r, angle2_r)

# def calculate_punch_speed(theta_curr, theta_prev, L1, L2, dt):
def calculate_punch_speed(theta, omega, L1, L2):
    th1, th2 = theta
    w1, w2 = omega
    
    s1, c1 = np.sin(th1), np.cos(th1)
    s12, c12 = np.sin(th1 + th2), np.cos(th1 + th2)

    J = np.array([
        [-L1 * s1 - L2 * s12, -L2 * s12],
        [ L1 * c1 + L2 * c12,  L2 * c12]
    ])
    
    v_fist = np.dot(J, np.array([w1, w2]))
    return np.linalg.norm(v_fist)

In [7]:
def create_arm_kalman_filter():
    kf = cv2.KalmanFilter(4, 2)
    
    kf.measurementMatrix = np.array([
        [1, 0, 0, 0],
        [0, 1, 0, 0]
    ], dtype=np.float32)
    
    kf.transitionMatrix = np.array([
        [1, 0, 1, 0],
        [0, 1, 0, 1],
        [0, 0, 1, 0],
        [0, 0, 0, 1]
    ], dtype=np.float32)
    
    kf.processNoiseCov = np.eye(4, dtype=np.float32) * 0.03
    
    kf.measurementNoiseCov = np.eye(2, dtype=np.float32) * 0.1
    
    kf.errorCovPost = np.eye(4, dtype=np.float32) * 1.0
    
    return kf

def update_kalman(kf, angles, dt, is_initialized):
    th1, th2 = angles
    meas = np.array([[np.float32(th1)], [np.float32(th2)]])
    
    if not is_initialized:
        kf.statePost = np.array([[np.float32(th1)], [np.float32(th2)], [0.0], [0.0]], dtype=np.float32)
        return (th1, th2), (0.0, 0.0), True
        
    kf.transitionMatrix = np.array([
        [1, 0, dt,  0],
        [0, 1,  0, dt],
        [0, 0,  1,  0],
        [0, 0,  0,  1]
    ], dtype=np.float32)

    kf.predict()
    estimate = kf.correct(meas)

    filtered_theta = (float(estimate[0, 0]), float(estimate[1, 0]))
    filtered_omega = (float(estimate[2, 0]), float(estimate[3, 0]))
    
    return filtered_theta, filtered_omega, True

In [ ]:
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print('Проблема с открытием веб-камеры')

kf_l = create_arm_kalman_filter()
kf_r = create_arm_kalman_filter()
is_initialized_l = False
is_initialized_r = False

while cap.isOpened():
    read_ok, frame = cap.read()
    if not read_ok:
            print('Проблема с потоком')
            break
    frame = cv2.flip(frame, 1)
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(frame_rgb)
    
    h,w,_=frame.shape
    current_time = time.time()
    dt=current_time - prev_time
    prev_time=current_time
    dt=max(min(dt,0.1),0.001)
    
    #yolo блок
    yolo_results=yolo_model(frame, conf=0.7, verbose=False)[0]

    for box in yolo_results.boxes:
         x1,y1,x2,y2 = map(int,box.xyxy[0])
         confidence = float(box.conf[0])
         cls_id = int(box.cls[0])
         class_name = yolo_model.names[cls_id]
         label = f'{class_name} {confidence}'
         cv2.rectangle(frame,(x1,y1),(x2,y2),(255,0,0),2)
         cv2.putText(frame,label,(x1,max(y1-10,20)),cv2.FONT_HERSHEY_SIMPLEX,0.6,(255,0,0),2)


    if results.pose_landmarks:
        # mp_drawing.draw_landmarks(
        #      frame,
        #      results.pose_landmarks
        # )
        #мне нужны 11 12 13 14 19 20
        l_shoulder = results.pose_landmarks.landmark[11]
        r_shoulder = results.pose_landmarks.landmark[12]
        l_elbow = results.pose_landmarks.landmark[13]
        r_elbow = results.pose_landmarks.landmark[14]
        l_wrist = results.pose_landmarks.landmark[15]
        r_wrist = results.pose_landmarks.landmark[16]
        l_hip = results.pose_landmarks.landmark[23]
        r_hip = results.pose_landmarks.landmark[24]
        cords = {
            'l_shoulder':np.array([l_shoulder.x*w,l_shoulder.y*h,l_shoulder.z*w]),
            'r_shoulder':np.array([r_shoulder.x*w,r_shoulder.y*h,r_shoulder.z*w]),
            'l_elbow':np.array([l_elbow.x*w,l_elbow.y*h,l_elbow.z*w]),
            'r_elbow':np.array([r_elbow.x*w,r_elbow.y*h,r_elbow.z*w]),
            'l_wrist':np.array([l_wrist.x*w,l_wrist.y*h,l_wrist.z*w]),
            'r_wrist':np.array([r_wrist.x*w,r_wrist.y*h,r_wrist.z*w]),
            'l_hip':np.array([l_hip.x*w,l_hip.y*h,l_hip.z*w]),
            'r_hip':np.array([r_hip.x*w,r_hip.y*h,r_hip.z*w])
            
        }
        # cords = {
        #             'l_shoulder':np.array([l_shoulder.x,l_shoulder.y,l_shoulder.z]),
        #             'r_shoulder':np.array([r_shoulder.x,r_shoulder.y,r_shoulder.z]),
        #             'l_elbow':np.array([l_elbow.x,l_elbow.y,l_elbow.z]),
        #             'r_elbow':np.array([r_elbow.x,r_elbow.y,r_elbow.z]),
        #             'l_wrist':np.array([l_wrist.x,l_wrist.y,l_wrist.z]),
        #             'r_wrist':np.array([r_wrist.x,r_wrist.y,r_wrist.z])
        #         }
        theta_l, theta_r = calculate_angle(cords)
        # cv2.putText(frame,f'{np.degrees(theta1)}',(int(l_wrist.x*w),int(l_wrist.y*h)),cv2.FONT_HERSHEY_COMPLEX, 1, (0, 0, 255))
        L1_l=np.linalg.norm(cords['l_shoulder']-cords['l_elbow'])
        L2_l=np.linalg.norm(cords['l_elbow']-cords['l_wrist'])
        L1_r=np.linalg.norm(cords['r_shoulder']-cords['r_elbow'])
        L2_r=np.linalg.norm(cords['r_elbow']-cords['r_wrist'])

        filt_theta_l, omega_l, is_initialized_l = update_kalman(kf_l, theta_l, dt, is_initialized_l)
        filt_theta_r, omega_r, is_initialized_r = update_kalman(kf_r, theta_r, dt, is_initialized_r)
        # cv2.putText(frame,f'{np.degrees(theta2)}',(int(r_wrist.x*w),int(r_wrist.y*h)),cv2.FONT_HERSHEY_COMPLEX, 1, (0, 0, 255))
        # current_time=time.time()
        # dt=current_time-prev_time
        speed_l = calculate_punch_speed(filt_theta_l, omega_l, L1_l, L2_l)
        speed_r = calculate_punch_speed(filt_theta_r, omega_r, L1_r, L2_r)
        # alpha = 0.05
        # smoothed_speed_l = 0.0
        # smoothed_speed_r = 0.0

        # smoothed_speed_l = alpha * speed_l + (1 - alpha) * smoothed_speed_l
        # smoothed_speed_r = alpha * speed_r + (1 - alpha) * smoothed_speed_r
        cv2.putText(frame, f'Скорость правой: {int(speed_l)}', (int(l_wrist.x*w), int(l_wrist.y*h - 30)), cv2.FONT_HERSHEY_COMPLEX, 1, (0, 255, 0), 2)
        cv2.putText(frame, f'Скорость левой: {int(speed_r)}', (int(r_wrist.x*w), int(r_wrist.y*h - 30)), cv2.FONT_HERSHEY_COMPLEX, 1, (0, 255, 0), 2)
        # prev_time = current_time
        # prev_theta_l = theta_l
        # prev_theta_r = theta_r
        # if prev_theta1 is not None and dt>0:
        #      w1=(theta1-prev_theta1)/dt
        #      w2=(theta2 - prev_theta2) / dt
            
            
        # else:
        #     w1, w2 = 0.0, 0.0
        #     prev_time=current_time
        # --- Левая рука ---
        if not is_punching_l:
            if speed_l > PUNCH_START:
                is_punching_l = True
                curr_peak_l = speed_l
        else:
            if speed_l > curr_peak_l:
                curr_peak_l = speed_l
            elif speed_l < PUNCH_END:
                is_punching_l = False
                last_peak_l = curr_peak_l
                timer_l = current_time + DISPLAY_SECS
                curr_peak_l = 0.0

        if not is_punching_r:
            if speed_r > PUNCH_START:
                is_punching_r = True
                curr_peak_r = speed_r
        else:
            if speed_r > curr_peak_r:
                curr_peak_r = speed_r
            elif speed_r < PUNCH_END:
                is_punching_r = False
                last_peak_r = curr_peak_r
                timer_r = current_time + DISPLAY_SECS
                curr_peak_r = 0.0

        # Определяем, что выводить (активный удар -> последний сохраненный -> 0)
        disp_l = curr_peak_l if is_punching_l else (last_peak_l if current_time < timer_l else 0.0)
        disp_r = curr_peak_r if is_punching_r else (last_peak_r if current_time < timer_r else 0.0)

        # Вывод со смещением по Y (50 для левой, 90 для правой)
        cv2.putText(frame, f"Max right:  {int(disp_l)}", (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)
        cv2.putText(frame, f"Max left: {int(disp_r)}", (30, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)
    else:
        is_initialized_l = False
        is_initialized_r = False
        cv2.putText(frame,'Позы не обнаружено',
                     (int(w*0.2),int(h*0.2)),
                     cv2.FONT_HERSHEY_COMPLEX,
                       1, 
                       (0, 0, 255)
                    )
    cv2.imshow('video', frame)
    if cv2.waitKey(20)==27:
        break
cap.release()
cv2.destroyAllWindows()